# 17 — Video Understanding: Frame Sampling & Temporal Reasoning

**Orbit:** Multimodal Systems · **Difficulty:** ★★★☆ Intermediate-Advanced · **Reading time:** ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/17_video_understanding_frame_sampling_and_temporal_reasoning.ipynb)

**Prerequisites:** [05 — Image Prompting & Visual Reasoning](05_image_prompting_and_visual_reasoning.ipynb) · [09 — Captioning as a Text Bridge](09_captioning_as_a_text_bridge.ipynb) · [16 — Multimodal Agents](16_multimodal_agents_across_speech_and_vision.ipynb)

**What you will learn:**
- How to extract informative frames from video using multiple sampling strategies.
- How to stay within a vision-language model's token budget.
- How to caption individual frames and aggregate them into video-level understanding.
- How temporal ordering of frames affects question-answering quality.
- A simple evaluation framework for temporal reasoning accuracy.

## Learning Objectives

By the end of this notebook you will be able to:

1. **Extract frames** from video using uniform, keyframe, and scene-change sampling strategies.
2. **Understand the token budget equation:** `frames × patches_per_frame × tokens_per_patch`.
3. **Use Qwen2.5-VL** to describe individual video frames with natural language.
4. **Aggregate frame-level descriptions** into a coherent video-level summary.
5. **Perform temporal QA** over ordered frame sequences and observe how shuffling degrades answers.
6. **Evaluate temporal accuracy** using rank-correlation metrics such as Kendall's τ.

## Why Video Understanding Is Hard

A video is nothing more than a sequence of images played back rapidly enough to 
create the illusion of motion. A modest 60-second clip recorded at 30 fps contains 
**1,800 individual frames**. If each frame is encoded into 256 vision tokens — a 
typical figure for models that split an image into a 16×16 patch grid — the full 
video would require roughly **460,800 tokens**. That comfortably exceeds the context 
window of every publicly available large language model today.

This is why a naive "process every frame" strategy is impractical. Even if a model 
could ingest half a million tokens, the compute cost would be enormous and most of 
those tokens would be redundant — consecutive frames in a static scene are nearly 
identical. The fundamental challenge of video understanding is therefore one of 
**information selection**: choosing the smallest set of frames that still preserves 
the visual story while fitting inside a fixed token budget. Getting this selection 
right is the difference between a model that understands a video and one that either 
runs out of context or drowns in irrelevant detail.

The rest of this notebook builds the machinery for that selection process step by step.

## Frame Sampling Strategies

There are four main families of frame sampling, each with different trade-offs:

### (a) Uniform Sampling
Extract every *N*-th frame — for instance one frame per second. This is dead simple 
to implement and produces evenly spaced snapshots. The downside is that it treats 
all moments as equally important: a ten-second static establishing shot receives 
the same number of frames as a one-second burst of rapid action.

### (b) Keyframe / I-Frame Extraction
Video codecs (H.264, H.265) already mark certain frames as *keyframes* (I-frames) 
that store a complete image rather than a diff. Extracting these is computationally 
cheap because the codec has already decided they represent significant visual states. 
However, keyframe placement is codec-dependent and may not align with semantic events.

### (c) Scene-Change Detection
Compare consecutive frames using pixel-level or histogram-level differences. When the 
difference exceeds a threshold the algorithm declares a scene change and keeps that 
frame. This works very well for edited content such as movies and TV shows where hard 
cuts introduce large visual discontinuities.

### (d) Adaptive / Content-Aware Sampling
Combine a low-rate uniform baseline with denser sampling during segments flagged as 
visually interesting — high optical-flow magnitude, scene changes, or detected actions. 
This is the most flexible strategy and generally yields the best quality per token, 
but it also requires the most engineering effort.

In [ ]:
# ── Environment Setup ──────────────────────────────────────────
!pip install -q transformers torch qwen-vl-utils opencv-python-headless pillow bitsandbytes accelerate

In [ ]:
import json, math, os, time, textwrap, warnings, random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import torch

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "font.size": 9})

print(f"PyTorch {torch.__version__}  •  CUDA available: {torch.cuda.is_available()}")
print("Dependencies ready.")

## Helper — Synthetic Video Generator

Since we cannot bundle a real video file inside a notebook, we generate a **synthetic 
10-second clip at 30 fps** (300 frames). The video has four distinct scenes with 
smooth colour transitions and abrupt scene changes at known timestamps. Each frame 
carries an overlay showing its timestamp, making it easy to verify that our sampling 
strategies pick the right moments.

In [ ]:
VIDEO_PATH = "synthetic_video.mp4"
FPS = 30
DURATION = 10  # seconds
WIDTH, HEIGHT = 320, 240
TOTAL_FRAMES = FPS * DURATION

# Define 4 scenes with distinct base colours (BGR for OpenCV)
SCENES = [
    {"start": 0.0, "end": 2.5, "colour": (40, 40, 200), "label": "Red Scene"},
    {"start": 2.5, "end": 5.0, "colour": (200, 120, 30), "label": "Blue Scene"},
    {"start": 5.0, "end": 7.5, "colour": (30, 180, 50),  "label": "Green Scene"},
    {"start": 7.5, "end": 10.0, "colour": (50, 200, 220), "label": "Yellow Scene"},
]
SCENE_BOUNDARIES = [s["start"] for s in SCENES[1:]]  # ground-truth changes

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_PATH, fourcc, FPS, (WIDTH, HEIGHT))

for i in range(TOTAL_FRAMES):
    t = i / FPS
    # Find current scene
    scene = SCENES[0]
    for s in SCENES:
        if s["start"] <= t < s["end"]:
            scene = s
            break
    # Add a slow brightness oscillation within the scene
    brightness = int(30 * math.sin(2 * math.pi * t / DURATION))
    base = np.array(scene["colour"], dtype=np.int16)
    colour = tuple(np.clip(base + brightness, 0, 255).astype(np.uint8).tolist())
    frame = np.full((HEIGHT, WIDTH, 3), colour, dtype=np.uint8)
    # Overlay timestamp and scene label
    label_text = f"{t:05.2f}s  {scene['label']}"
    cv2.putText(frame, label_text, (10, HEIGHT // 2), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (255, 255, 255), 2, cv2.LINE_AA)
    writer.write(frame)

writer.release()
print(f"Wrote {TOTAL_FRAMES} frames ({DURATION}s @ {FPS} fps) -> {VIDEO_PATH}")
print(f"Ground-truth scene boundaries: {SCENE_BOUNDARIES}")

## Uniform Sampling

The simplest strategy: grab one frame per second. For a 10-second clip this gives us 
10 frames — a drastic reduction from the original 300.

In [ ]:
def uniform_sample(video_path, target_fps=1):
    """Return list of (timestamp_sec, PIL.Image) at the target fps."""
    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    step = max(1, int(round(src_fps / target_fps)))
    frames, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            t = idx / src_fps
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append((round(t, 2), Image.fromarray(rgb)))
        idx += 1
    cap.release()
    return frames

uniform_frames = uniform_sample(VIDEO_PATH, target_fps=1)
print(f"Uniform sampling: {len(uniform_frames)} frames")

# Display grid
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for ax, (t, img) in zip(axes.flat, uniform_frames):
    ax.imshow(img)
    ax.set_title(f"{t:.1f}s", fontsize=8)
    ax.axis("off")
fig.suptitle("Uniform Sampling (1 fps)", fontsize=11)
plt.tight_layout()
plt.show()

## Scene-Change Detection

We compare consecutive frames by computing the **mean absolute difference** (MAD) 
of their pixel values. When the MAD exceeds a threshold, we declare a scene change. 
This is a simple but effective heuristic for content with hard cuts. For real-world 
use you could replace MAD with histogram intersection, structural similarity (SSIM), 
or a learned scene-boundary detector. The key insight is that scene boundaries carry 
the highest information density — the visual content changes dramatically, so these 
frames are almost always worth keeping.

In [ ]:
def detect_scene_changes(video_path, threshold=30.0):
    """Return list of (timestamp, PIL.Image) at detected scene boundaries."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    prev_gray = None
    diffs, timestamps = [], []
    frames_all = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
        t = idx / fps
        if prev_gray is not None:
            mad = np.mean(np.abs(gray - prev_gray))
            diffs.append(mad)
            timestamps.append(t)
            frames_all.append((t, frame))
        else:
            diffs.append(0.0)
            timestamps.append(t)
            frames_all.append((t, frame))
        prev_gray = gray
        idx += 1
    cap.release()

    # Always include first frame; then frames exceeding threshold
    scene_frames = [(frames_all[0][0],
                     Image.fromarray(cv2.cvtColor(frames_all[0][1], cv2.COLOR_BGR2RGB)))]
    scene_times = [0.0]
    for i, d in enumerate(diffs):
        if d > threshold and i > 0:
            t_val, bgr = frames_all[i]
            scene_frames.append((round(t_val, 2),
                                 Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))))
            scene_times.append(round(t_val, 2))
    return scene_frames, timestamps, diffs, scene_times

scene_frames, ts, diffs, detected_boundaries = detect_scene_changes(VIDEO_PATH, threshold=30)
print(f"Detected {len(scene_frames)} scene-change keyframes at t = {detected_boundaries}")
print(f"Ground truth boundaries: {SCENE_BOUNDARIES}")

In [ ]:
# Visualise the difference signal
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(ts, diffs, linewidth=0.8, label="Frame MAD")
ax.axhline(30, color="red", linestyle="--", linewidth=0.7, label="Threshold")
for b in SCENE_BOUNDARIES:
    ax.axvline(b, color="green", alpha=0.5, linewidth=1.2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Mean Abs Diff")
ax.set_title("Frame-to-Frame Difference with Scene Boundaries (green)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Adaptive Sampling

Adaptive sampling merges a coarse uniform baseline (e.g., one frame every two seconds) 
with extra frames around detected scene changes. This ensures we have broad temporal 
coverage *and* capture the critical transition moments that carry the most information.

In [ ]:
def adaptive_sample(video_path, base_fps=0.5, scene_threshold=30.0, context_window=0.3):
    """Combine uniform baseline with scene-boundary keyframes."""
    base_frames = uniform_sample(video_path, target_fps=base_fps)
    sc_frames, _, _, sc_times = detect_scene_changes(video_path, threshold=scene_threshold)

    # Merge: keep all base frames + scene frames not already near a base frame
    base_times = {t for t, _ in base_frames}
    merged = list(base_frames)
    for t, img in sc_frames:
        if all(abs(t - bt) > context_window for bt in base_times):
            merged.append((t, img))
    merged.sort(key=lambda x: x[0])
    return merged

adaptive_frames = adaptive_sample(VIDEO_PATH)
print(f"Adaptive sampling: {len(adaptive_frames)} frames")
print(f"Timestamps: {[t for t, _ in adaptive_frames]}")

## Token Budget Calculator

Every frame a VLM processes consumes tokens. The total cost follows a simple formula:

$$\text{total\_tokens} = n_{\text{frames}} \times \text{patches\_per\_frame} \times \text{tokens\_per\_patch}$$

For Qwen2.5-VL with a 320×240 image and default settings, each frame produces roughly 
256 vision tokens. The table below compares our three strategies against the budget.

In [ ]:
PATCHES = 256  # approximate tokens per frame for a 320x240 image

strategies = {
    "All frames (30 fps)": TOTAL_FRAMES,
    "Uniform (1 fps)": len(uniform_frames),
    "Scene-change": len(scene_frames),
    "Adaptive": len(adaptive_frames),
}

rows = []
for name, n in strategies.items():
    tokens = n * PATCHES
    rows.append({"Strategy": name, "Frames": n, "Tokens": f"{tokens:,}",
                 "Fits 8k ctx": "Y" if tokens < 8192 else "N"})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 3.5))
names = list(strategies.keys())
counts = [v * PATCHES for v in strategies.values()]
colours = ["#e74c3c" if c > 8192 else "#2ecc71" for c in counts]
ax.barh(names, counts, color=colours)
ax.axvline(8192, color="black", linestyle="--", linewidth=1, label="8k context")
ax.set_xlabel("Vision Tokens")
ax.set_title("Token Cost by Sampling Strategy")
ax.legend()
plt.tight_layout()
plt.show()

## Temporal Reasoning — Why Order Matters

Vision-language models perceive each frame independently — they have no built-in notion 
of "before" or "after". Temporal reasoning must therefore be constructed explicitly: 
we present the model with an **ordered sequence** of frame descriptions, each tagged 
with its timestamp, and ask questions that require understanding the progression of 
events.

Consider the difference between *"What colour appeared after blue?"* and *"What colour 
appeared in the video?"* The first question requires temporal ordering; the second does 
not. If we shuffle the frame descriptions and re-ask the first question, the model will 
either guess or hallucinate — providing direct evidence that its answers depended on the 
order of the input.

This observation has a practical implication: always present frame captions in 
chronological order and always include explicit timestamps. The model cannot recover 
temporal information that was never provided.

## Load Qwen2.5-VL (4-bit Quantised)

We load the vision-language model in 4-bit precision using `BitsAndBytesConfig` to fit 
comfortably within T4 GPU memory. The Google Drive cache avoids re-downloading weights 
across Colab sessions.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/models"
except Exception:
    CACHE_DIR = "./models"

os.makedirs(CACHE_DIR, exist_ok=True)

VLM_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    cache_dir=CACHE_DIR,
)
vlm_processor = AutoProcessor.from_pretrained(VLM_ID, cache_dir=CACHE_DIR)
print(f"Loaded {VLM_ID} in 4-bit  •  device: {vlm.device}")

In [ ]:
# ── VLM Helper: describe_frame ──────────────────────────────────
def describe_frame(image: Image.Image, prompt: str = "Describe this video frame in one sentence.") -> str:
    """Send a single image + text prompt to Qwen2.5-VL and return the generated text."""
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]}
    ]
    text_input = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_input], images=[image], return_tensors="pt", padding=True
    ).to(vlm.device)
    with torch.no_grad():
        ids = vlm.generate(**inputs, max_new_tokens=120)
    output_ids = ids[:, inputs["input_ids"].shape[1]:]
    return vlm_processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

# Quick smoke test with the first uniform frame
test_cap = describe_frame(uniform_frames[0][1])
print(f"Smoke test caption: {test_cap}")

## Frame-Level Captioning

We loop through the uniformly sampled frames and generate a one-sentence description 
for each. The resulting `(timestamp, caption)` pairs form the raw material for all 
downstream temporal reasoning.

In [ ]:
captions = []
for t, img in uniform_frames:
    cap = describe_frame(img)
    captions.append((t, cap))
    print(f"  t={t:5.1f}s | {cap}")

caption_df = pd.DataFrame(captions, columns=["time_s", "caption"])
caption_df

## Video-Level Summary via Text Aggregation

The simplest way to go from frame captions to a video summary is to **concatenate all 
captions in chronological order** and ask a language model to synthesise them. We load 
**Qwen3-8B** in 4-bit for this text-only aggregation step.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

TEXT_MODEL_ID = "Qwen/Qwen3-8B"

text_tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID, cache_dir=CACHE_DIR)
text_llm = AutoModelForCausalLM.from_pretrained(
    TEXT_MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    cache_dir=CACHE_DIR,
)
print(f"Loaded {TEXT_MODEL_ID} in 4-bit  •  device: {text_llm.device}")

In [ ]:
def ask_text_model(prompt: str, max_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text_input = text_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = text_tok(text_input, return_tensors="pt").to(text_llm.device)
    with torch.no_grad():
        ids = text_llm.generate(**inputs, max_new_tokens=max_tokens)
    output_ids = ids[:, inputs["input_ids"].shape[1]:]
    return text_tok.decode(output_ids[0], skip_special_tokens=True).strip()

timeline = "\n".join(f"[{t:.1f}s] {c}" for t, c in captions)
summary_prompt = (
    "Below are timestamped descriptions of video frames in chronological order.\n\n"
    f"{timeline}\n\n"
    "Write a concise 2-3 sentence summary of what happens in this video."
)
video_summary = ask_text_model(summary_prompt)
print("=== Video Summary ===")
print(textwrap.fill(video_summary, 90))

## Temporal Question Answering

Temporal QA asks questions whose answers depend on the *order* or *timing* of events. 
We provide the ordered frame captions as context and pose questions that require the 
model to reason about "before", "after", and sequence. Good temporal QA performance 
is evidence that the model is actually using the timeline, not just pattern-matching 
keywords in the captions.

In [ ]:
TEMPORAL_QUESTIONS = [
    "What is the first scene colour shown in the video?",
    "What happened immediately after the blue scene?",
    "List the scene colours in the order they appear.",
    "What is the last scene shown in the video?",
]

print("=== Temporal QA (ordered captions) ===")
ordered_answers = []
for q in TEMPORAL_QUESTIONS:
    prompt = (
        f"Video frame descriptions (chronological):\n{timeline}\n\n"
        f"Question: {q}\nAnswer concisely."
    )
    ans = ask_text_model(prompt, max_tokens=80)
    ordered_answers.append(ans)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

## Shuffle Experiment — Does Order Matter?

We now **randomly shuffle** the same frame captions, removing all temporal structure, 
and repeat the same questions. If temporal ordering truly matters, the answers should 
degrade noticeably.

In [ ]:
shuffled_captions = list(captions)
random.seed(42)
random.shuffle(shuffled_captions)
shuffled_timeline = "\n".join(f"[{t:.1f}s] {c}" for t, c in shuffled_captions)

print("=== Temporal QA (SHUFFLED captions) ===")
shuffled_answers = []
for q in TEMPORAL_QUESTIONS:
    prompt = (
        f"Video frame descriptions:\n{shuffled_timeline}\n\n"
        f"Question: {q}\nAnswer concisely."
    )
    ans = ask_text_model(prompt, max_tokens=80)
    shuffled_answers.append(ans)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

# Side-by-side comparison
comp_df = pd.DataFrame({
    "Question": TEMPORAL_QUESTIONS,
    "Ordered Answer": ordered_answers,
    "Shuffled Answer": shuffled_answers,
})
comp_df

## Evaluating Temporal Accuracy

How do we *measure* whether a model's temporal reasoning is correct? One robust approach 
is **Kendall's tau (τ)**, a rank-correlation statistic. Given a ground-truth ordering of 
events and the model's predicted ordering, τ ranges from −1 (perfectly reversed) to +1 
(perfectly matched). A value near 0 means the model's ordering is essentially random. 
Below we implement a lightweight evaluator that extracts an event ordering from the 
model's response and scores it against the known scene sequence.

In [ ]:
from scipy.stats import kendalltau

GROUND_TRUTH_ORDER = ["Red", "Blue", "Green", "Yellow"]

def extract_order(text: str, labels=None):
    """Extract the order in which labels appear in text."""
    if labels is None:
        labels = GROUND_TRUTH_ORDER
    positions = []
    for lab in labels:
        idx = text.lower().find(lab.lower())
        positions.append(idx if idx != -1 else 9999)
    ranked = sorted(range(len(labels)), key=lambda i: positions[i])
    return [labels[i] for i in ranked]

# Use the ordering question (index 2)
order_q_idx = 2
pred_ordered = extract_order(ordered_answers[order_q_idx])
pred_shuffled = extract_order(shuffled_answers[order_q_idx])

gt_ranks = list(range(len(GROUND_TRUTH_ORDER)))
ord_ranks = [GROUND_TRUTH_ORDER.index(c) if c in GROUND_TRUTH_ORDER else -1
             for c in pred_ordered]
shuf_ranks = [GROUND_TRUTH_ORDER.index(c) if c in GROUND_TRUTH_ORDER else -1
              for c in pred_shuffled]

tau_ordered, _ = kendalltau(gt_ranks, ord_ranks)
tau_shuffled, _ = kendalltau(gt_ranks, shuf_ranks)

print(f"Ground truth order : {GROUND_TRUTH_ORDER}")
print(f"Predicted (ordered) : {pred_ordered}  ->  tau = {tau_ordered:.2f}")
print(f"Predicted (shuffled): {pred_shuffled}  ->  tau = {tau_shuffled:.2f}")

## Keyframe Quality Comparison

Do scene-change keyframes produce more informative captions than uniformly sampled 
frames? We caption the scene-change frames and compare the average caption length and 
vocabulary diversity as rough proxies for informativeness.

In [ ]:
scene_captions = []
for t, img in scene_frames:
    cap = describe_frame(img)
    scene_captions.append((t, cap))
    print(f"  t={t:5.1f}s | {cap}")

def caption_stats(caps):
    lengths = [len(c.split()) for _, c in caps]
    vocab = set()
    for _, c in caps:
        vocab.update(c.lower().split())
    return {"n_frames": len(caps), "avg_words": round(np.mean(lengths), 1),
            "vocab_size": len(vocab)}

stats = pd.DataFrame([
    {"Strategy": "Uniform", **caption_stats(captions)},
    {"Strategy": "Scene-change", **caption_stats(scene_captions)},
])
print("\n", stats.to_string(index=False))

## Exercises

### Exercise 1 — Motion-Based Adaptive Sampler
Implement an adaptive sampler that uses **optical flow magnitude** as the activity 
signal. Extract more frames during high-motion segments and fewer during static 
segments. Use `cv2.calcOpticalFlowFarneback` to compute dense optical flow between 
consecutive frames, then set a dynamic sampling rate proportional to the mean flow 
magnitude.

### Exercise 2 — Temporal QA Benchmark
Build a small benchmark of **5 temporal questions** about the synthetic video with 
known ground-truth answers. Score the VLM's answers using exact-match accuracy and 
a simple word-overlap similarity metric. Compare performance when using uniform 
sampling vs. scene-change sampling as the frame source.

### Exercise 3 — Minimum Frame Budget
What is the **minimum number of frames** needed to correctly answer all your temporal 
questions? Perform a binary search over frame counts (1 to 10) using uniform sampling. 
At each count, run the full QA pipeline and record accuracy. Plot accuracy vs. frame 
budget and identify the knee of the curve.

In [ ]:
# ── Exercise 1 Starter Code ────────────────────────────────────
def optical_flow_sampler(video_path, base_fps=0.5, flow_threshold=2.0):
    """Sample more frames when optical flow is high.

    TODO:
      1. Read frames from video_path.
      2. Compute dense optical flow between consecutive frames
         using cv2.calcOpticalFlowFarneback.
      3. Compute mean flow magnitude per frame.
      4. Always keep frames at `base_fps` intervals.
      5. Additionally keep any frame where mean magnitude > flow_threshold.
      6. Return list of (timestamp, PIL.Image).
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = max(1, int(round(fps / base_fps)))
    prev_gray = None
    results = []
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        t = idx / fps
        keep = (idx % step == 0)  # uniform baseline

        if prev_gray is not None:
            # TODO: compute optical flow and mean magnitude
            # flow = cv2.calcOpticalFlowFarneback(
            #     prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0
            # )
            # mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            # if np.mean(mag) > flow_threshold:
            #     keep = True
            pass

        if keep:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results.append((round(t, 2), Image.fromarray(rgb)))
        prev_gray = gray
        idx += 1

    cap.release()
    return results

# Uncomment to test after implementing:
# flow_frames = optical_flow_sampler(VIDEO_PATH)
# print(f"Optical-flow sampler: {len(flow_frames)} frames")

## Key Takeaways

| # | Insight |
|---|---|
| 1 | Raw video is far too token-expensive for any current VLM — sampling is mandatory. |
| 2 | Uniform sampling is a strong baseline but wastes budget on static scenes. |
| 3 | Scene-change detection concentrates budget on high-information moments. |
| 4 | Adaptive sampling gives the best quality-per-token by combining strategies. |
| 5 | Temporal QA requires chronologically ordered captions with explicit timestamps. |
| 6 | Kendall's τ provides a quantitative measure of temporal reasoning accuracy. |

## References

1. **Video-LLaVA** — Lin et al., 2023. [arXiv:2311.10122](https://arxiv.org/abs/2311.10122)
2. **TimeChat** — Ren et al., 2023. [arXiv:2312.02051](https://arxiv.org/abs/2312.02051)
3. **Qwen2.5-VL Technical Report** — Bai et al., 2025. [arXiv:2502.13923](https://arxiv.org/abs/2502.13923)
4. **OpenCV VideoCapture docs** — [docs.opencv.org](https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html)
5. **Kendall's tau** — Kendall, 1938. [Wikipedia](https://en.wikipedia.org/wiki/Kendall_rank_correlation_coefficient)

---

← [16 — Multimodal Agents across Speech & Vision](16_multimodal_agents_across_speech_and_vision.ipynb) · 
[18 — Video Grounding, Summarisation & Event Extraction](18_video_grounding_summarization_and_event_extraction.ipynb) →